# Excel File Manipulation with pandas


## Reading Excel Files with pandas


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel(
    "xl/stores.xlsx",
    sheet_name="2019",
    skiprows=1,
    usecols="B:F",
    engine="calamine",
)
df

In [ ]:
df.info()

In [ ]:
def fix_missing(x):
    return False if x in ["", "MISSING"] else x

In [ ]:
df = pd.read_excel(
    "xl/stores.xlsx",
    sheet_name="2019",
    skiprows=1,
    usecols="B:F",
    converters={"Flagship": fix_missing},
    engine="calamine",
)
df

In [ ]:
# The Flagship column now has Dtype "bool"
df.info()

In [ ]:
sheets = pd.read_excel(
    "xl/stores.xlsx",
    sheet_name=["2019", "2020"],
    skiprows=1,
    usecols=["Store", "Employees"],
    engine="calamine",
)
sheets["2019"].head(2)

In [ ]:
df = pd.read_excel(
    "xl/stores.xlsx",
    sheet_name=0,
    skiprows=2,
    skipfooter=3,
    usecols="B:C,F",
    header=None,
    names=["Branch", "Employee_Count", "Is_Flagship"],
    engine="calamine",
)
df

In [ ]:
df = pd.read_excel(
    "xl/stores.xlsx",
    sheet_name="2019",
    skiprows=1,
    usecols="B,C,F",
    skipfooter=2,
    na_values="MISSING",
    keep_default_na=False,
    engine="calamine",
)
df

In [ ]:
with pd.ExcelFile("xl/stores.xls") as f:
    df1 = pd.read_excel(f, "2019", skiprows=1, usecols="B:F", nrows=2)
    df2 = pd.read_excel(f, "2020", skiprows=1, usecols="B:F", nrows=2)

df1

In [ ]:
stores = pd.ExcelFile("xl/stores.xlsx", engine="calamine")
stores.sheet_names

In [ ]:
url = "https://raw.githubusercontent.com/fzumstein/python-for-excel/2e/xl/stores.xlsx"
pd.read_excel(
    url,
    skiprows=1,
    usecols="B:E",
    nrows=2,
    engine="calamine",
)

### Reading in Parallel

In [ ]:
import multiprocessing
from functools import partial

def read_excel_parallel(filename, sheet_name=0, engine=None):
    if sheet_name is None:
        # Read all sheets
        with pd.ExcelFile(filename, engine=engine) as xlfile:
            sheet_names = xlfile.sheet_names

    # Create a partial function with filename and engine pre-filled.
    # A partial function sets certain arguments of a function,
    # creating a new function (read_func) with fewer parameters.
    read_func = partial(pd.read_excel, filename, engine=engine)

    with multiprocessing.Pool() as pool:
        # map calls read_func for each sheet in parallel
        dfs = pool.map(read_func, sheet_names)

    # zip pairs sheet names with DataFrames into tuples
    return dict(zip(sheet_names, dfs))

In [ ]:
%%time
data = pd.read_excel("xl/big.xlsx", sheet_name=None, engine="calamine")

In [ ]:
data["Sheet1"].info()

In [ ]:
%%time
data = read_excel_parallel(
    "xl/big.xlsx",
    sheet_name=None,
    engine="calamine",
)

In [ ]:
data["Sheet1"].info()

## Writing Excel Files with pandas


In [ ]:
import numpy as np
import datetime as dt

In [ ]:
data = {
    "Dates": [
        dt.datetime(2020, 1, 1, 10, 13),
        dt.datetime(2020, 1, 2),
        dt.datetime(2020, 1, 2),
    ],
    "Floats": [2.222, np.nan, np.inf],
    "Integers": [1, 2, 3],
    "Booleans": [True, False, True],
}
df = pd.DataFrame(data)
df.index.name = "index"
df

In [ ]:
df.to_excel(
    "written_with_pandas.xlsx",
    sheet_name="Output",
    startrow=1,
    startcol=1,
    index=True,
    header=True,
    na_rep="=NA()",  # Defaults to empty cell
    inf_rep="=NA()",  # Defaults to "inf"
)

In [ ]:
with pd.ExcelWriter("written_with_pandas2.xlsx") as writer:
    df.to_excel(writer, sheet_name="Sheet1", startrow=1, startcol=1)
    df.to_excel(writer, sheet_name="Sheet1", startrow=10, startcol=1)
    df.to_excel(writer, sheet_name="Sheet2")